**Downloading data**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('allstate-claims-severity')

print("Path to competition files:", path)

In [2]:
from pathlib import Path
import pandas as pd

def load_housing_data(x):
    return pd.read_csv(x, low_memory=False)

train= load_housing_data("/kaggle/input/competitions/allstate-claims-severity/train.csv")
test= load_housing_data("/kaggle/input/competitions/allstate-claims-severity/test.csv")

In [16]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 188318 entries, 0 to 188317
Columns: 132 entries, id to loss
dtypes: float64(15), int64(1), object(116)
memory usage: 189.7+ MB


**Rows that contain any missing value**

In [15]:
train[train.isna().any(axis=1)]

,id,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,...,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13,cont14,loss


In [11]:
corr_matrix = train.corr(numeric_only=True)
corr_matrix["loss"].sort_values(ascending=False)

loss      1.000000
cont2     0.141528
cont7     0.119799
cont3     0.111053
cont11    0.099806
cont12    0.098724
cont6     0.040967
cont8     0.030508
cont10    0.020236
cont14    0.019298
cont9     0.014456
id       -0.001122
cont13   -0.004022
cont1    -0.010237
cont5    -0.011355
cont4    -0.035831
Name: loss, dtype: float64

In [3]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.base import clone
from sklearn.dummy import DummyRegressor

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# ------------------------------------------------------------------
# 0. Informational checks
# ------------------------------------------------------------------
print("lightgbm:", lgb.__version__, "| xgboost:", xgb.__version__)

# ------------------------------------------------------------------
# 1. Features / target
# ------------------------------------------------------------------
drop_cols    = [c for c in ["id", "loss"] if c in train.columns]
feature_cols = [c for c in train.columns if c not in drop_cols]

X = train[feature_cols].copy()
y = train["loss"].copy()

Y_NONNEG = bool(y.min() >= 0)
print(f"target min={y.min():.4f} max={y.max():.4f} "
      f"mean={y.mean():.4f} std={y.std():.4f} median={y.median():.4f}")
print(f"clipping predictions at 0: {Y_NONNEG}")

def clip_pred(p):
    return np.clip(p, 0, None) if Y_NONNEG else p

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"num_cols={len(num_cols)} cat_cols={len(cat_cols)}")

# ------------------------------------------------------------------
# 2. Shared preprocessing for the sklearn-style models
#    (LightGBM + XGBoost). CatBoost gets raw frame + cat feature names.
# ------------------------------------------------------------------
pre = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        # no scaling needed for trees, but harmless; keep it off for speed
    ]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value",
                                  unknown_value=-1)),
    ]), cat_cols),
])

# ------------------------------------------------------------------
# 3. Model definitions
# ------------------------------------------------------------------
lgbm_reg = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    device_type="gpu",  # uncomment if you have a GPU-enabled build
)

xgb_reg = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",   # "gpu_hist" if GPU available
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    objective="reg:absoluteerror",  # MAE objective (XGBoost >= 1.7)
)

cat_reg = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=7,
    l2_leaf_reg=3.0,
    loss_function="MAE",       # CatBoost supports MAE directly
    random_seed=42,
    verbose=False,
    task_type="GPU",         # uncomment if GPU available
)

models = {
    "lightgbm": Pipeline([("pre", pre), ("reg", lgbm_reg)]),
    "xgboost":  Pipeline([("pre", pre), ("reg", xgb_reg)]),
    # CatBoost: pass raw frame + cat feature indices to use native cats
    "catboost": cat_reg,
}

# ------------------------------------------------------------------
# 4. Cross-validation
# ------------------------------------------------------------------
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
folds = list(kf.split(X))

results = {name: {"train": [], "val": [], "oof": np.zeros(len(y))}
           for name in models}
base_scores = []

# Pre-compute cat feature indices for CatBoost (positional in feature_cols)
cat_idx = [feature_cols.index(c) for c in cat_cols]

for fold, (tr_idx, val_idx) in enumerate(folds, start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # median baseline
    base = DummyRegressor(strategy="median").fit(X_tr, y_tr)
    bs = mean_absolute_error(y_val, clip_pred(base.predict(X_val)))
    base_scores.append(bs)

    line = f"Fold {fold:2d} | baseline={bs:.4f}"

    for name, est in models.items():
        if name == "catboost":
            m = clone(est)
            m.fit(X_tr, y_tr, cat_features=cat_idx)
        else:
            m = clone(est)
            m.fit(X_tr, y_tr)

        tr_pred = clip_pred(m.predict(X_tr))
        vl_pred = clip_pred(m.predict(X_val))

        tr_mae = mean_absolute_error(y_tr, tr_pred)
        vl_mae = mean_absolute_error(y_val, vl_pred)

        results[name]["train"].append(tr_mae)
        results[name]["val"].append(vl_mae)
        results[name]["oof"][val_idx] = vl_pred

        line += f" | {name}={vl_mae:.4f}"

    print(line)

# ------------------------------------------------------------------
# 5. Summary table
# ------------------------------------------------------------------
def pooled_mae(name):
    return mean_absolute_error(y, results[name]["oof"])

summary = []
for name in models:
    summary.append({
        "model": name,
        "mean_train_mae": np.mean(results[name]["train"]),
        "std_train_mae":  np.std(results[name]["train"]),
        "mean_val_mae":   np.mean(results[name]["val"]),
        "std_val_mae":    np.std(results[name]["val"]),
        "pooled_oof_mae": pooled_mae(name),
        "oof_mae_over_std": pooled_mae(name) / y.std(),
        "impr_vs_baseline_%":
            (np.mean(base_scores) - np.mean(results[name]["val"]))
            / np.mean(base_scores) * 100,
    })

summary_df = pd.DataFrame(summary).sort_values("pooled_oof_mae")
print("\n=== Model comparison (sorted by pooled OOF MAE) ===")
print(summary_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print(f"\nMedian-baseline mean val MAE: "
      f"{np.mean(base_scores):.4f} +/- {np.std(base_scores):.4f}")

# ------------------------------------------------------------------
# 6. Optional: simple equal-weight blend of the three OOF predictions
# ------------------------------------------------------------------
blend_oof = np.mean([results[n]["oof"] for n in models], axis=0)
print(f"Equal-weight blend pooled OOF MAE: "
      f"{mean_absolute_error(y, blend_oof):.4f}")

lightgbm: 4.6.0 | xgboost: 3.4.1
target min=0.6700 max=121012.2500 mean=3037.3377 std=2904.0862 median=2115.5700
clipping predictions at 0: True
num_cols=14 cat_cols=116


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature n

Fold  1 | baseline=1796.9615 | lightgbm=1173.9781 | xgboost=1134.5749 | catboost=1789.9522


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
Default metric period is 5 because MAE is/are not implemented for GPU


Fold  2 | baseline=1819.9197 | lightgbm=1188.2626 | xgboost=1151.0154 | catboost=1812.8436


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
Default metric period is 5 because MAE is/are not implemented for GPU


Fold  3 | baseline=1820.3204 | lightgbm=1187.1646 | xgboost=1150.7031 | catboost=1813.2993


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
Default metric period is 5 because MAE is/are not implemented for GPU


Fold  4 | baseline=1826.4480 | lightgbm=1178.8326 | xgboost=1144.0225 | catboost=1819.4314


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
Default metric period is 5 because MAE is/are not implemented for GPU


Fold  5 | baseline=1781.6902 | lightgbm=1173.5350 | xgboost=1133.2953 | catboost=1774.6672

=== Model comparison (sorted by pooled OOF MAE) ===
   model  mean_train_mae  std_train_mae  mean_val_mae  std_val_mae  pooled_oof_mae  oof_mae_over_std  impr_vs_baseline_%
 xgboost        994.5380         1.2882     1142.7222       7.6080       1142.7223            0.3935             36.8336
lightgbm        984.2614         1.4468     1180.3546       6.2991       1180.3546            0.4064             34.7534
catboost       1802.0024         4.2295     1802.0387      16.9784       1802.0388            0.6205              0.3886

Median-baseline mean val MAE: 1809.0680 +/- 16.9856
Equal-weight blend pooled OOF MAE: 1236.3028
